In [10]:
import sys
from pathlib import Path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader
from torchvision import models 
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score
sys.path.append(str(Path("..")))
from src.dataset import train_data,train_path,train_indices,val_indices, val_loader, ood_loader,ood_path,transform
from src.sampler import BalancedBatchSampler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
import numpy as np
from torchvision import datasets,transforms

In [11]:
MODEL_PATH = Path("../05_02_UnFreeze_4.pth")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

TEST_DIRS = {
    "test0": Path("../data/test/test0"),
    "test1": Path("../data/test/test1"),
    "test1_2": Path("../data/test/test1_2"),
    "test2": Path("../data/test/test2"),
    "test3": Path("../data/test/test3"),
    "test4": Path("../data/test/test4"),
}

best_threshold = -18.040495
ENERGY_THRESHOLD = float(best_threshold)

ENERGY_TESTS = {"test1", "test1_2", "test2", "test3", "test4"}

BATCH_SIZE = 64
NEYSAN_NAMES = {"neysan"}

cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [12]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True
)

num_classes = len(train_data.classes)

model = models.convnext_tiny(
    weights=models.ConvNeXt_Tiny_Weights.DEFAULT
)
model.classifier[2] = nn.Linear(
    model.classifier[2].in_features,
    num_classes
)

model = model.to(device)

model.load_state_dict(checkpoint)
model.to(device)
model.eval()


known_classes = list(train_data.classes)
class_to_idx = {
    name.lower(): i
    for i, name in enumerate(known_classes)
}

print("Known classes:", known_classes)
print("Energy threshold:", ENERGY_THRESHOLD)



all_results = {}

for test_name, test_path in TEST_DIRS.items():

    print("\n" + "=" * 65)
    print("DATASET:", test_name)
    print("=" * 65)

    if not test_path.exists():
        print("Path not found:", test_path.resolve())
        print("Update TEST_DIRS and rerun.")
        continue

    test_dataset =datasets.ImageFolder(
        root=str(test_path),
        transform=transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    print("Number of images:", len(test_dataset))
    print("Folder classes:", test_dataset.classes)

    true_labels = []
    predicted_labels = []
    energy_scores = []
    review_samples = []



    sample_index = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            logits = model(images)


            predictions = torch.argmax(logits, dim=1)

            if test_name in ENERGY_TESTS:
                energies = -torch.logsumexp(logits, dim=1)
            else:
                energies = None

            batch_size = len(images)

            for i in range(batch_size):

                dataset_index = sample_index + i
                image_path, folder_label = test_dataset.samples[
                    dataset_index
                ]

                true_name = test_dataset.classes[folder_label]
                predicted_name = known_classes[
                    predictions[i].item()
                ]

                true_labels.append(true_name)
                predicted_labels.append(predicted_name)

                if energies is not None:

                    energy = energies[i].item()
                    energy_scores.append(energy)

                    if energy > ENERGY_THRESHOLD:

                        review_samples.append({
                            "path": image_path,
                            "folder_label": true_name,
                            "predicted_class": predicted_name,
                            "energy": energy
                        })

            sample_index += batch_size



    known_indices = [
        i for i, name in enumerate(true_labels)
        if name.lower() in class_to_idx
    ]

    y_true_known = np.array([
        class_to_idx[true_labels[i].lower()]
        for i in known_indices
    ])

    y_pred_known = np.array([
        class_to_idx[predicted_labels[i].lower()]
        for i in known_indices
    ])

    if len(known_indices) > 0:

        known_accuracy = accuracy_score(
            y_true_known,
            y_pred_known
        )

        known_precision = precision_score(
            y_true_known,
            y_pred_known,
            labels=list(range(len(known_classes))),
            average="macro",
            zero_division=0
        )

        known_recall = recall_score(
            y_true_known,
            y_pred_known,
            labels=list(range(len(known_classes))),
            average="macro",
            zero_division=0
        )

        known_f1 = f1_score(
            y_true_known,
            y_pred_known,
            labels=list(range(len(known_classes))),
            average="macro",
            zero_division=0
        )

        print("\n--- Standard 8-class results ---")
        print(f"Accuracy:  {known_accuracy:.4f}")
        print(f"Macro Precision: {known_precision:.4f}")
        print(f"Macro Recall:    {known_recall:.4f}")
        print(f"Macro F1:        {known_f1:.4f}")

        print("\nConfusion matrix (known classes):")
        print(confusion_matrix(
            y_true_known,
            y_pred_known,
            labels=list(range(len(known_classes)))
        ))


    if test_name in ENERGY_TESTS:

        # Samples with a known folder label
        known_total = len(known_indices)

        known_review_count = sum(
            i not in [
                j for j, name in enumerate(true_labels)
                if name.lower() in NEYSAN_NAMES
            ]
            for i in range(len(review_samples))
        )

        # Use folder labels to identify explicit Nissan folders.
        nissan_indices = [
            i for i, name in enumerate(true_labels)
            if name.lower() in NEYSAN_NAMES
        ]

        review_paths = {
            item["path"] for item in review_samples
        }

        known_review_count = sum(
            true_labels[i].lower() not in NEYSAN_NAMES
            and test_dataset.samples[i][1] >= 0
            and test_dataset.samples[i][0] in review_paths
            for i in range(len(true_labels))
        )

        print("\n--- Energy / NEED_REVIEW results ---")
        print("Energy threshold:", ENERGY_THRESHOLD)
        print("Total flagged for review:", len(review_samples))

        if known_total > 0:
            print(
                "Known samples flagged for review:",
                known_review_count
            )
            print(
                "Known review rate:",
                f"{known_review_count / known_total:.2%}"
            )

        if len(nissan_indices) > 0:
            nissan_reviewed = sum(
                true_labels[i].lower() in NEYSAN_NAMES
                and test_dataset.samples[i][0] in review_paths
                for i in range(len(true_labels))
            )

            print("Nissan samples:", len(nissan_indices))
            print("Nissan sent to NEED_REVIEW:", nissan_reviewed)
            print(
                "Nissan detection rate:",
                f"{nissan_reviewed / len(nissan_indices):.2%}"
            )
        else:
            print(
                "Nissan detection rate cannot be calculated from "
                "folder labels alone for this dataset."
            )

    if test_name in ENERGY_TESTS:

        # Samples accepted by Energy
        accepted_indices = [
            i for i in range(len(true_labels))
            if test_dataset.samples[i][0] not in {
                item["path"] for item in review_samples
            }
        ]

        # NEED_REVIEW statistics
        total_samples = len(true_labels)
        review_count = len(review_samples)
        accepted_count = total_samples - review_count

        print("\n--- After Energy filtering ---")
        print("Total samples:", total_samples)
        print("NEED_REVIEW:", review_count)
        print(
            "NEED_REVIEW rate:",
            f"{review_count / total_samples:.2%}"
        )
        print("Accepted samples:", accepted_count)
        print(
            "Acceptance rate:",
            f"{accepted_count / total_samples:.2%}"
        )

        # Accuracy among accepted samples, using folder labels.
        # Explicit Nissan folders are excluded because Nissan
        # is not one of the model's 8 known classes.
        accepted_known_indices = [
            i for i in accepted_indices
            if true_labels[i].lower() not in NEYSAN_NAMES
        ]

        if accepted_known_indices:

            accepted_true = [
                class_to_idx[true_labels[i].lower()]
                for i in accepted_known_indices
            ]

            accepted_pred = [
                class_to_idx[predicted_labels[i].lower()]
                for i in accepted_known_indices
            ]

            accepted_accuracy = accuracy_score(
                accepted_true,
                accepted_pred
            )

            print(
                "Accuracy among accepted known samples:",
                f"{accepted_accuracy:.4f}"
            )

        # Known acceptance rate
        known_indices_for_acceptance = [
            i for i, name in enumerate(true_labels)
            if name.lower() not in NEYSAN_NAMES
        ]

        if known_indices_for_acceptance:

            known_accepted = sum(
                i in accepted_indices
                for i in known_indices_for_acceptance
            )

            known_acceptance_rate = (
                known_accepted / len(known_indices_for_acceptance)
            )

            print(
                "Known acceptance rate:",
                f"{known_acceptance_rate:.2%}"
            )

    all_results[test_name] = {
        "true_labels": true_labels,
        "predicted_labels": predicted_labels,
        "energy_scores": energy_scores,
        "review_samples": review_samples
    }

print("\nAll available test datasets processed.")

Known classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
Energy threshold: -18.040495

DATASET: test0
Number of images: 961
Folder classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

--- Standard 8-class results ---
Accuracy:  0.9677
Macro Precision: 0.9457
Macro Recall:    0.9752
Macro F1:        0.9579

Confusion matrix (known classes):
[[  9   0   0   0   0   0   0   0]
 [  0 146   0   0   0   0   0   0]
 [  1   0 159  16   0   0   0   0]
 [  0   0   6 138   0   0   0   0]
 [  0   0   0   0  52   0   0   0]
 [  1   0   0   0   0 198   1   0]
 [  0   0   0   0   1   4 188   0]
 [  1   0   0   0   0   0   0  40]]

DATASET: test1
Number of images: 1081
Folder classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'neysan', 'savari', 'taxi', 'vanet']

--- Standard 8-class results ---
Accuracy:  0.9678
Macro Precision: 0.9457
Macro Recall:    0.9753
Macro F1:        0.9580

Confusion matrix 

KeyboardInterrupt: 